# Notebook 2: The Stress Recovery Clock

Measuring how long it takes for seismicity statistics to recover after M7+ earthquakes,
using b-value recovery time constants, Modified Omori Law fitting, and inter-event time analysis.

In [1]:
import matplotlib
matplotlib.use('Agg')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import estimate_b_value, rolling_b_value
from src.omori import fit_omori, compute_recovery_fraction, fit_recovery_exponential
from src.spatial import haversine, events_in_radius
from src.interevent import compute_interevent_times, rolling_cv
from src.plotting import setup_style, save_figure, plot_recovery_gallery

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")

Loaded 681,450 events


## 2.1 Mainshock Selection

In [3]:
# Select M7+ events
m7plus = df[df["mag"] >= 7.0].copy().sort_values("time").reset_index(drop=True)
print(f"M7+ events before declustering: {len(m7plus)}")

# Decluster: keep only the largest event within 100 km / 30 days
declustered = []
used = set()
for i, row in m7plus.iterrows():
    if i in used:
        continue
    # Find all M7+ within 100km and 30 days
    cluster = [i]
    for j, row2 in m7plus.iterrows():
        if j <= i or j in used:
            continue
        dist = haversine(row["latitude"], row["longitude"], row2["latitude"], row2["longitude"])
        dt = abs((pd.Timestamp(row2["time"]) - pd.Timestamp(row["time"])).total_seconds()) / 86400
        if dist <= 100 and dt <= 30:
            cluster.append(j)
    # Keep the largest
    best = max(cluster, key=lambda k: m7plus.loc[k, "mag"])
    declustered.append(m7plus.loc[best])
    used.update(cluster)

mainshocks = pd.DataFrame(declustered).reset_index(drop=True)
print(f"Mainshocks after declustering: {len(mainshocks)}")

M7+ events before declustering: 387


Mainshocks after declustering: 362


## 2.2 Recovery Analysis

In [4]:
recovery_results = []

for idx, ms in mainshocks.iterrows():
    ms_time = pd.Timestamp(ms["time"])
    if ms_time.tzinfo is None:
        ms_time = ms_time.tz_localize("UTC")
    ms_mag = ms["mag"]
    ms_lat = ms["latitude"]
    ms_lon = ms["longitude"]

    # Aftershock radius: R = 10^(0.5*M - 1.8) km
    radius_km = 10 ** (0.5 * ms_mag - 1.8)

    # Collect aftershocks (1000 days)
    t_end = ms_time + pd.Timedelta(days=1000)
    aftershocks = events_in_radius(df, ms_lat, ms_lon, radius_km)
    aftershocks = aftershocks[(aftershocks["time"] > ms_time) & (aftershocks["time"] <= t_end)]

    if len(aftershocks) < 30:
        continue

    mc = estimate_mc(aftershocks["mag"].values)

    # Pre-mainshock baseline (5 years before)
    t_pre_start = ms_time - pd.Timedelta(days=5*365)
    pre_events = events_in_radius(df, ms_lat, ms_lon, radius_km)
    pre_events = pre_events[(pre_events["time"] >= t_pre_start) & (pre_events["time"] < ms_time)]
    pre_above_mc = pre_events[pre_events["mag"] >= mc]

    if len(pre_above_mc) < 10:
        b_baseline = 1.0
    else:
        try:
            b_baseline, _ = estimate_b_value(pre_above_mc["mag"].values, mc)
        except ValueError:
            b_baseline = 1.0

    # Rolling b-value in aftershock zone
    as_above = aftershocks[aftershocks["mag"] >= mc]
    if len(as_above) < 60:
        continue

    rolling = rolling_b_value(as_above["time"].values, as_above["mag"].values,
                              mc=mc, window_size=min(50, len(as_above)//2), step=10)

    if len(rolling) < 3:
        continue

    # Recovery fraction — handle tz-naive center_time from rolling_b_value
    b_series = rolling["b_value"].values
    t_days = []
    for t in rolling["center_time"]:
        ct = pd.Timestamp(t)
        if ct.tzinfo is None:
            ct = ct.tz_localize("UTC")
        t_days.append((ct - ms_time).total_seconds() / 86400)
    t_days = np.array(t_days)

    recovery_frac = compute_recovery_fraction(b_series, b_baseline)
    fit_result = fit_recovery_exponential(recovery_frac, t_days)

    # Omori fit
    omori = fit_omori(aftershocks["time"], ms_time)

    result = {
        "name": ms.get("place", f"M{ms_mag:.1f}"),
        "time": ms_time,
        "mag": ms_mag,
        "lat": ms_lat,
        "lon": ms_lon,
        "n_aftershocks": len(aftershocks),
        "mc": mc,
        "b_baseline": b_baseline,
        "t_days": t_days,
        "b_values": b_series,
        "recovery_frac": recovery_frac,
        "tau_b": fit_result["tau_b"] if fit_result else np.nan,
        "recovery_r2": fit_result["r_squared"] if fit_result else np.nan,
        "omori_p": omori["p"] if omori else np.nan,
        "omori_c": omori["c"] if omori else np.nan,
        "omori_r2": omori["r_squared"] if omori else np.nan,
    }
    recovery_results.append(result)

print(f"Successfully analyzed {len(recovery_results)} mainshock sequences")

Successfully analyzed 139 mainshock sequences


## 2.3 Recovery Curve Gallery

In [5]:
# Select 9 well-constrained examples (best R²)
well_constrained = [r for r in recovery_results if r["recovery_r2"] > 0.3]
well_constrained.sort(key=lambda r: r["mag"], reverse=True)
gallery_data = well_constrained[:9]

if len(gallery_data) > 0:
    fig = plot_recovery_gallery(gallery_data, n_rows=3, n_cols=3)
    save_figure(fig, "02_recovery_gallery")
    plt.show()
    print(f"Showing {len(gallery_data)} best-constrained recovery curves")
else:
    print("No well-constrained recovery curves found (R² > 0.3)")

Showing 9 best-constrained recovery curves


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_15596/3700293659.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4 τ_b vs. Magnitude

In [6]:
results_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ["t_days", "b_values", "recovery_frac"]}
                           for r in recovery_results])

well = results_df[results_df["recovery_r2"] > 0.3].copy()
print(f"Well-constrained recoveries: {len(well)} / {len(results_df)}")

if len(well) > 5:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(well["mag"], well["tau_b"], c=well["recovery_r2"],
               cmap="viridis", s=50, alpha=0.7)
    plt.colorbar(ax.collections[0], ax=ax, label="R²")

    # Regression line
    from scipy import stats as sp_stats
    slope, intercept, r, p, se = sp_stats.linregress(well["mag"], well["tau_b"])
    x_fit = np.linspace(well["mag"].min(), well["mag"].max(), 50)
    ax.plot(x_fit, slope * x_fit + intercept, "r--", linewidth=1.5,
            label=f"Linear fit (R²={r**2:.2f}, p={p:.3f})")

    ax.set_xlabel("Mainshock Magnitude")
    ax.set_ylabel("τ_b (days)")
    ax.set_title("b-value Recovery Time vs. Mainshock Magnitude")
    ax.legend()
    save_figure(fig, "02_tau_b_vs_magnitude")
    plt.show()

Well-constrained recoveries: 17 / 139


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_15596/674014113.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.5 Global Map of Recovery Times

In [7]:
if len(well) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))
    sc = ax.scatter(well["lon"], well["lat"], c=well["tau_b"], cmap="plasma",
                    s=well["mag"]**2, alpha=0.7, edgecolors="k", linewidth=0.3)
    plt.colorbar(sc, ax=ax, label="τ_b (days)")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")
    ax.set_title("M7+ Mainshocks Colored by b-value Recovery Time")
    ax.set_aspect("equal")
    save_figure(fig, "02_global_recovery_map")
    plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_15596/2616920228.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.6 Omori p-value Distribution

In [8]:
valid_p = results_df["omori_p"].dropna()
if len(valid_p) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(valid_p, bins=30, color="#457B9D", alpha=0.7, edgecolor="white")
    ax.axvline(1.0, color="#E63946", linewidth=2, linestyle="--", label="p = 1.0")
    ax.set_xlabel("Omori p-value")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of Modified Omori Law p-values")
    ax.legend()
    save_figure(fig, "02_omori_p_histogram")
    plt.show()
    print(f"Median p = {valid_p.median():.2f}, Mean p = {valid_p.mean():.2f}")

Median p = 1.86, Mean p = 1.98


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_15596/2146716869.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook measured post-mainshock recovery dynamics for M7+ earthquakes worldwide:

- **b-value recovery**: Tracked how the Gutenberg-Richter b-value evolves after large mainshocks, fitting an exponential recovery model to estimate the characteristic time constant τ_b.
- **Modified Omori Law**: Fit aftershock decay rates to obtain p-values characterizing how quickly aftershock productivity diminishes.
- **Scaling with magnitude**: Examined whether larger mainshocks produce longer recovery times, as expected from stress perturbation scaling.
- **Spatial patterns**: Mapped recovery times globally to identify regional variations in crustal healing rates.

These recovery time constants provide a quantitative measure of the Earth's "stress memory" — the timescale over which the crust returns to its background statistical state after a major perturbation.